#### Purpose

Evaluate the best trained Telco Churn neural network on the untouched test dataset.

The model was trained, selected using validation loss, restored to its best checkpoint, and persisted to MLflow in 04_train_neural_network.

The goal is to measure how well the selected model generalizes to customers that were not used for training or model selection.

The evaluation flow is:

``` text

MLflow Run
    ↓
Load Best PyTorch Model
    ↓
Load Fitted Preprocessor
    ↓
Recreate Same Test Split
    ↓
Transform Test Features
    ↓
Forward Propagation
    ↓
Logits
    ↓
Test Loss
    ↓
Sigmoid
    ↓
Probabilities
    ↓
Classification Threshold
    ↓
Predicted Classes
    ↓
Evaluation Metrics

```

##### Production Design

Notebook 05 should not depend on Notebook 04 memory.

``` text

04_train_neural_network
        ↓
MLflow
  ├── trained model
  ├── fitted preprocessor
  ├── parameters
  ├── metrics
  ├── signature
  └── input example
        ↓
05_evaluate_model
        ↓
Load persisted artifacts
        ↓
Evaluate independently

```

The test dataset is used only after training and model selection are complete.

#### Technologies Used

- Databricks
- PyTorch
- MLflow
- scikit-learn
- Pandas
- NumPy
- joblib
- Matplotlib

Important components:

- mlflow.pytorch.load_model
- mlflow.artifacts.download_artifacts
- joblib.load
- torch.sigmoid
- model.eval()
- torch.no_grad()
- accuracy_score
- precision_score
- recall_score
- f1_score
- roc_auc_score
- confusion_matrix

#### Input

The notebook requires:

- MLflow Run ID
- Gold Delta table
- Project configuration

Latest successful model run: 6b15c39ebfb04e3a9c5803acf2f279a9

The model expects: 45 processed numerical features and produces:1 raw churn logit per customer

#### Output

This notebook will produce:

- Test Loss
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Average Precision
- Confusion Matrix - TN, FP, FN, TP
- ROC Curve
- Precision-Recall Curve
- Educational Threshold Sensitivity Analysis

#### Evaluation Architecture

``` text

MLflow Run
    ↓
Load Model
    ↓
Load Preprocessor
    ↓

Gold Delta Table
    ↓
Recreate Same Train / Val / Test Split
    ↓
Select Test Set Only
    ↓
Apply Saved Preprocessor
    ↓
X_test_tensor
    ↓

model.eval()
    ↓
torch.no_grad()
    ↓
Forward Propagation
    ↓
Logits
    ↓
BCEWithLogitsLoss
    ↓
Test Loss

and separately:

Logits
    ↓
Sigmoid
    ↓
Probabilities
    ↓
Threshold
    ↓
Predicted Classes
    ↓
Metrics

```

The key point is: The test set is only used now, after training and model selection are complete.

##### 1. Load Project Configuration

In [0]:
%run ./00_project_config

##### 2. Import Evaluation Components

In [0]:
import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from sklearn.model_selection import train_test_split

##### 3. Define the MLflow Run ID

In [0]:
RUN_ID = "6b15c39ebfb04e3a9c5803acf2f279a9"

``` text
RUN_ID
   ↓
Which training experiment?
   ↓
Which exact model?
   ↓
Which exact preprocessor?

```

##### 4. Load Persisted PyTorch Model

In [0]:
model_uri = f"runs:/{RUN_ID}/model"

model = mlflow.pytorch.load_model(
    model_uri
)

model.eval()

print(model)

``` text

MLflow
   ↓
Saved model artifact
   ↓
load_model()
   ↓
model

```

model.train() → training mode

model.eval() → evaluation / inference mode

##### 5. Load Persisted Fitted Preprocessor

In [0]:
preprocessor_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID,
    artifact_path="preprocessing/telco_nn_preprocessor.joblib",
)

preprocessor = joblib.load(
    preprocessor_path
)


``` text

Saved fitted preprocessor
        ↓
Same medians
Same means
Same standard deviations
Same one-hot categories

```
This is very important.

We are not calling: build_preprocessor() and then refitting it.

That would be wrong for evaluation.

Instead:

Training → fit preprocessor

Evaluation
→ load fitted preprocessor
→ transform only

It contains learned preprocessing state such as:

SimpleImputer
→ training median

StandardScaler
→ training means
→ training standard deviations

OneHotEncoder
→ learned categories

##### 6. Load Gold Dataset

In [0]:
df = (
    spark.table(GOLD_TABLE)
    .toPandas()
)

print(
    "Dataset shape:",
    df.shape,
)

##### 7. Create X and y

In [0]:
y = df[TARGET_COLUMN]

X = df.drop(
    columns=COLUMNS_TO_DROP,
    errors="ignore",
)

print(
    "X shape:",
    X.shape,
)

print(
    "y shape:",
    y.shape,
)

##### 8. Recreate the Original Train / Validation / Test Split

In [0]:
#Use exactly the same configuration as training:

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=VALIDATION_TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

##### 9. Verify Test Split

In [0]:
print(
    "X_test shape:",
    X_test.shape,
)

print(
    "y_test shape:",
    y_test.shape,
)

In [0]:
#Check class distribution:
print(
    y_test.value_counts(
        normalize=True
    )
)

##### 10. Transform Test Features

In [0]:
#Use the already-fitted preprocessor:

X_test_processed = preprocessor.transform(
    X_test
)

print(
    "Processed test shape:",
    X_test_processed.shape,
)

##### 11. Convert Test Data to PyTorch Tensors

In [0]:
X_test_tensor = torch.tensor(
    X_test_processed,
    dtype=torch.float32,
)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32,
).unsqueeze(1)

In [0]:
print(
    "X_test_tensor:",
    X_test_tensor.shape,
)

print(
    "y_test_tensor:",
    y_test_tensor.shape,
)

##### 12. Run Test Inference

In [0]:
model.eval()

with torch.no_grad():

    test_logits = model(
        X_test_tensor
    )

In [0]:
print(
    "Test logits shape:",
    test_logits.shape,
)

##### 13. Calculate Test Loss

In [0]:
criterion = nn.BCEWithLogitsLoss()

test_loss = criterion(
    test_logits,
    y_test_tensor,
)

print(
    "Test Loss:",
    round(
        test_loss.item(),
        4,
    ),
)

##### 14. Convert Logits to Probabilities

In [0]:
test_probabilities = torch.sigmoid(
    test_logits
)

##### 15. Convert Probabilities to Predicted Classes

In [0]:
test_predictions = (
    test_probabilities
    >= CLASSIFICATION_THRESHOLD
).float()

##### 16. Convert Tensors to NumPy for scikit-learn Metrics

In [0]:
y_true = (
    y_test_tensor
    .cpu()
    .numpy()
    .ravel()
)

y_pred = (
    test_predictions
    .cpu()
    .numpy()
    .ravel()
)

y_prob = (
    test_probabilities
    .cpu()
    .numpy()
    .ravel()
)

##### 17. Calculate Test Metrics

In [0]:
accuracy = accuracy_score(
    y_true,
    y_pred,
)

precision = precision_score(
    y_true,
    y_pred,
)

recall = recall_score(
    y_true,
    y_pred,
)

f1 = f1_score(
    y_true,
    y_pred,
)

roc_auc = roc_auc_score(
    y_true,
    y_prob,
)

average_precision = average_precision_score(
    y_true,
    y_prob,
)

In [0]:
print(
    "Test Loss:",
    round(test_loss.item(), 4),
)

print(
    "Accuracy:",
    round(accuracy, 4),
)

print(
    "Precision:",
    round(precision, 4),
)

print(
    "Recall:",
    round(recall, 4),
)

print(
    "F1 Score:",
    round(f1, 4),
)

print(
    "ROC-AUC:",
    round(roc_auc, 4),
)

print(
    "Average Precision:",
    round(average_precision, 4),
)

| Metric    | Test Result | Meaning                                                |
| --------- | ----------: | ------------------------------------------------------ |
| Test Loss |  **0.4193** | Overall BCE loss on unseen test customers              |
| Accuracy  |  **80.32%** | Overall percentage classified correctly                |
| Precision |  **70.39%** | Of customers predicted to churn, ~70% actually churned |
| Recall    |  **44.84%** | Of customers who actually churned, ~45% were detected  |
| F1        |  **54.78%** | Balance between precision and recall                   |
| ROC-AUC   |  **84.59%** | Good ability to rank churners above non-churners       |


##### 18. Calculate the Confusion Matrix

In [0]:
cm = confusion_matrix(
    y_true,
    y_pred,
)

print(cm)

``` text

                     Predicted

                  No Churn    Churn

Actual No Churn      TN         FP

Actual Churn         FN         TP

```

##### 19. Extract Confusion Matrix Values - TN, FP, FN, TP

In [0]:
tn, fp, fn, tp = cm.ravel()

print(
    "True Negatives:",
    tn,
)

print(
    "False Positives:",
    fp,
)

print(
    "False Negatives:",
    fn,
)

print(
    "True Positives:",
    tp,
)

##### What Do These Mean for Churn?

This is important because positive in our project means: Churn = 1

Therefore:

TRUE NEGATIVE (TN)

Actual: No Churn

Prediction: No Churn

✓ Correct

Business meaning: The customer stayed, and our model correctly predicted that they would stay.

FALSE POSITIVE (FP)

Actual: No Churn

Prediction: Churn

✗ Incorrect

Business meaning: The model flagged a customer as a churn risk, but the customer actually stayed.

FALSE NEGATIVE (FN)

Actual: Churn

Prediction: No Churn

✗ Incorrect

The customer actually churned, but our model failed to identify them as a churn risk.

TRUE POSITIVE (TP)

Actual: Churn

Prediction: Churn

✓ Correct

Business meaning: The customer actually churned and our model correctly identified them as being at risk.

These are customers for whom a retention strategy could potentially be useful.

##### 20. Verify Accuracy from Confusion Matrix

In [0]:
accuracy_check = (
    (tn + tp)
    / (tn + fp + fn + tp)
)

print(
    "Accuracy from confusion matrix:",
    round(
        accuracy_check,
        4,
    ),
)

##### 21. Verify Precision

In [0]:
precision_check = (
    tp
    / (tp + fp)
)

print(
    "Precision from confusion matrix:",
    round(
        precision_check,
        4,
    ),
)

##### 22. Verify Recall

In [0]:
recall_check = (
    tp
    / (tp + fn)
)

print(
    "Recall from confusion matrix:",
    round(
        recall_check,
        4,
    ),
)

##### 23. Plot Confusion Matrix

In [0]:
fig, ax = plt.subplots(
    figsize=(6, 5)
)

image = ax.imshow(
    cm
)

ax.set_xticks(
    [0, 1]
)

ax.set_yticks(
    [0, 1]
)

ax.set_xticklabels(
    [
        "No Churn",
        "Churn",
    ]
)

ax.set_yticklabels(
    [
        "No Churn",
        "Churn",
    ]
)

ax.set_xlabel(
    "Predicted Class"
)

ax.set_ylabel(
    "Actual Class"
)

ax.set_title(
    "Telco Churn - Confusion Matrix"
)

for row in range(2):

    for column in range(2):

        ax.text(
            column,
            row,
            cm[row, column],
            ha="center",
            va="center",
        )

fig.colorbar(
    image,
    ax=ax,
)

plt.show()

#### 24. Interpret Confusion Matrix

True Negative = 723
- → Customer stayed
- → Model predicted No Churn
- → Correct

False Positive = 53
- → Customer stayed
- → Model predicted Churn
- → Incorrect

False Negative = 155
- → Customer churned
- → Model predicted No Churn
- → Incorrect

True Positive = 126
- → Customer churned
- → Model predicted Churn
- → Correct


The main weakness at threshold 0.50 is:

Actual churners = 281

- 126 detected
- 155 missed

which produces: Recall = 44.84%

#####  Why Recall May Matter More for Churn

TRAINING → learns probability-producing model


THRESHOLD → business decision applied to probabilities

##### 25. Educational Threshold Sensitivity Analysis

The threshold analysis below demonstrates how classification thresholds affect precision and recall. Because it uses test predictions, it is diagnostic only and will not be used to select the production classification threshold.

In [0]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

thresholds = [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
]

threshold_results = []

for threshold in thresholds:

    predictions = (
        y_prob >= threshold
    ).astype(int)

    threshold_precision  = precision_score(
        y_true,
        predictions,
    )

    threshold_recall  = recall_score(
        y_true,
        predictions,
    )

    threshold_f1  = f1_score(
        y_true,
        predictions,
    )

    threshold_results.append(
        {
            "threshold": threshold,
            "precision": threshold_precision,
            "recall": threshold_recall,
            "f1_score": threshold_f1,
        }
    )

In [0]:
threshold_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_df
)

Your observed pattern:

Lower threshold
→ Recall increases
→ Precision decreases

Higher threshold
→ Precision increases
→ Recall decreases

This is the precision-recall tradeoff.


``` text

Same Trained Neural Network
            ↓
     Same Probabilities
            ↓
   ┌────────┼────────┐
   ↓        ↓        ↓
  0.30     0.40     0.50
   ↓        ↓        ↓
Different predicted classes
            ↓
Different Precision / Recall / F1

```

No weights change.

No backpropagation happens.

No Adam optimizer runs.

No retraining happens.

We're simply changing the rule that converts probability into a class.

As the threshold goes down:

``` text

Threshold ↓
    ↓
More customers classified as Churn
    ↓
True Positives likely ↑
False Positives likely ↑
    ↓
Recall usually ↑
Precision may ↓

```

As the threshold goes up:

``` text 

Threshold ↑
    ↓
Fewer customers classified as Churn
    ↓
False Positives likely ↓
True Positives may ↓
    ↓
Precision may ↑
Recall usually ↓

```

``` text

Threshold       0.30 ─────────────────────→ 0.60
                    ↑

Precision       56.3% ────────────────────→ 76.0%
                    ↑

Recall          68.7% ────────────────────→ 28.1%
                    ↓

```

##### 26. ROC Curve

In [0]:
fpr, tpr, roc_thresholds = roc_curve(
    y_true,
    y_prob,
)

- fpr → False Positive Rate
- tpr → True Positive Rate (Recall)
- roc_thresholds → classification thresholds

In [0]:
plt.figure(
    figsize=(7, 5)
)

plt.plot(
    fpr,
    tpr,
    label=f"Neural Network (AUC = {roc_auc:.4f})",
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier",
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve - Telco Churn Neural Network"
)

plt.legend()

plt.grid()

plt.show()

ROC-AUC = 0.8459   ← good ranking ability

Recall = 0.4484    ← many churners missed at threshold 0.50

##### 27. Precision-Recall Curve

In [0]:
pr_precision, pr_recall, pr_thresholds = (
    precision_recall_curve(
        y_true,
        y_prob,
    )
)

In [0]:
plt.figure(
    figsize=(7, 5)
)

plt.plot(
    pr_recall,
    pr_precision,
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Precision-Recall Curve - Telco Churn Neural Network"
)

plt.grid()

plt.show()

Lower threshold
      ↓
Predict more Churn
      ↓
Recall ↑
Precision may ↓


Higher threshold
      ↓
Predict fewer Churn
      ↓
Precision may ↑
Recall ↓

##### Why ROC and Precision-Recall Curves Are Different

The ROC curve compares:

``` text

True Positive Rate
        vs
False Positive Rate

```
where:

$$ TPR = \frac{TP}{TP+FN} $$

and:

$$ FPR = \frac{FP}{FP+TN} $$

So ROC asks:

How well can the model separate the positive and negative classes across thresholds?

The Precision-Recall curve compares:

``` text

Precision
    vs
Recall

```

where:

$$ Precision=\frac{TP}{TP+FP} $$ $$ Recall=\frac{TP}{TP+FN} $$

So it focuses more directly on:

When we try to detect churners, how many do we catch, and how trustworthy are those churn predictions?

For an imbalanced churn problem, this is particularly informative.

Precision
    ↓
One classification threshold


Average Precision
    ↓
Performance across thresholds

##### 28. Final Evaluation Results

In [0]:
evaluation_results = {
    "test_loss": test_loss.item(),
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "roc_auc": roc_auc,
    "average_precision": average_precision,
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
}

In [0]:
for metric, value in evaluation_results.items():

    if isinstance(
        value,
        float,
    ):
        print(
            f"{metric}: {value:.4f}"
        )

    else:
        print(
            f"{metric}: {value}"
        )

##### Key Learnings

1. Evaluation must use data that was not used to train the model or select the best checkpoint.

2. The trained neural network is loaded from MLflow rather than relying on notebook memory.

3. The same fitted preprocessing pipeline used during training must also be used during evaluation.

4. Evaluation uses preprocessor.transform(), never fit_transform().

5. model.eval() places the neural network into evaluation mode.

6. torch.no_grad() disables unnecessary gradient calculations during inference.

7. The neural network produces raw logits.

8. Sigmoid converts logits into churn probabilities.

9. The classification threshold converts probabilities into classes.

10. Accuracy measures the percentage of all predictions that are correct.

11. Precision asks:

    Of customers predicted to churn,
    how many actually churned?

12. Recall asks:

    Of all customers who actually churned,
    how many did the model detect?

13. F1 balances precision and recall.

14. ROC-AUC evaluates the model's ability to rank churn customers above non-churn customers across thresholds.

15. Average Precision summarizes Precision-Recall performance across thresholds.

16. The confusion matrix provides actual customer counts for:

    True Negatives
    False Positives
    False Negatives
    True Positives

17. The current threshold of 0.50 produces relatively good precision but lower recall.

18. Classification threshold selection is a business/model-selection decision and should normally be performed using validation data, not the final test set.

19. The test threshold table in this notebook is exploratory and educational only.

20. Neural-network performance should be compared with a simpler baseline before concluding that the additional complexity is worthwhile.

##### Conclusion

The trained Telco Churn neural network was loaded independently from MLflow and evaluated on the untouched test dataset.

The final test results were:

- Test Loss         = 0.4193
- Accuracy          = 0.8032
- Precision         = 0.7039
- Recall            = 0.4484
- F1 Score          = 0.5478
- ROC-AUC           = 0.8459
- Average Precision = 0.6544

The confusion matrix was:

``` text

                     Predicted
                 No Churn   Churn

Actual No Churn     723       53
Actual Churn        155      126

```

The neural network demonstrates good overall discrimination, as shown by ROC-AUC = 0.8459.

At the default classification threshold of 0.50, the model has relatively strong precision but lower recall, meaning that its churn predictions are fairly reliable while a significant number of actual churners remain undetected.

Threshold sensitivity analysis demonstrated the tradeoff between precision and recall. However, because the analysis used the final test set, it is treated as educational only and is not used to change the official model threshold.

The neural-network evaluation is now complete.

##### Next Notebook

##### 06_compare_ml_vs_neural_network

The next notebook will compare the neural network with the traditional Logistic Regression baseline.

The comparison will focus on:

- Model complexity
- Training approach
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Interpretability
- Training and maintenance complexity
- Whether deep learning provides meaningful additional value for this structured Telco dataset

The goal is not simply to determine which model has the highest score.

The goal is to understand whether the additional complexity of a neural network is justified compared with a simpler traditional machine-learning model.